# Signal Hunters R2 · v3 team stack (LLM_ops)

Two independent systems each score about 0.91 on the public leaderboard:

* **ours (v2):** canvas ResNet with EMA, full-train and noisy-student seeds, plus a LightGBM on 180 GPU physics features. Public 0.9099.
* **Piyush (v9/v10):** EfficientNet-B0/B2/B3 global-time canvases, plus a LightGBM stacker on level (scintillation) and track (ETI) features. Public 0.9158.

On the test set they disagree on about 5% of cadences, and almost all of that sits in two places:
**eti vs rfi_intermittent** (about 30% of predicted ETIs) and **scintillating_star vs noise / rfi_persistent**.
Different mistakes are exactly what a stacker can use.

This notebook is **CPU only (about 20 min)** and does no CNN training:

1. Load our `oof_and_test_probs.npz` and every Piyush `oofpks_<tag>.npy` / `test_<tag>.npy` pair it can find.
2. Train one LightGBM stacker on the **PKS-simulated out-of-fold rows** of all models together, plus our 180 physics features and Piyush's level and track features.
   Every input row is out-of-fold, so the stacker never sees a probability that was trained on its own label.
3. Decision layer: Saerens EM for the test label mix, then per-class logit biases. Biases are accepted only if they help on both cross-fitted halves.
4. Write several submissions, plus `pl.npz`: test cadences where the stack, our models and Piyush's models all agree with high confidence.
   `pl.npz` feeds the GPU student notebook (`v3_student`).

**Inputs to attach:** the competition data, and one dataset containing
`oof_and_test_probs.npz` and Piyush's whole `cache/` folder (`oofpks_*.npy`, `test_*.npy`, `lvl_*.npy`, `trk_*.npy`).
File names are found recursively, so the folder layout does not matter.
When the student run finishes, add its `out/*.npy` files to the same dataset and run this notebook again.
The new models are picked up automatically.

In [ ]:
import os, glob, time, json, warnings
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
warnings.filterwarnings('ignore')
T0 = time.time()

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")} | {(time.time() - T0) / 60:6.1f} min] {msg}', flush=True)

CONFIG = dict(
    SEEDS=[0, 1, 2],                 # LightGBM seeds per fold
    MAX_ROUNDS=1500, EARLY_STOP=60, LR=0.03, LEAVES=15, MIN_LEAF=40, FEAT_FRAC=0.5,
    OUR_A_TEST='ens',                # 'ens' = mean of all our Model A test models (CV + full-train + noisy student); 'cv5' = CV models only
    USE_OUR_FEATS=True,              # our 180 physics features
    USE_PIY_FEATS=True,              # Piyush level + track features (computed here if the files are missing)
    USE_PIY_STACK=True,              # also feed Piyush's own stacker OOF as an input
    EXCLUDE_TAGS=[],                 # Piyush model tags to leave out
    MIN_MODEL_F1=0.80,               # drop a model whose PKS OOF macro-F1 is below this (an unfinished run)
    PL_THR=0.90,                     # pseudo-label confidence threshold
    BIAS_GRID=np.round(np.arange(-1.0, 1.0001, 0.05), 3),
)
CLASSES = ['eti_candidate', 'frb', 'pulsar', 'scintillating_star', 'satellite_pass',
           'radar_chirp', 'rfi_intermittent', 'rfi_persistent', 'noise']
K = len(CLASSES)
C2I = {c: i for i, c in enumerate(CLASSES)}
SHORT = [c[:6] for c in CLASSES]

def mf1(yt, yp):
    return f1_score(yt, yp, average='macro', labels=list(range(K)))

def per_class(yt, yp):
    return f1_score(yt, yp, average=None, labels=list(range(K)))

def find(pattern):
    return sorted(glob.glob(os.path.join('/kaggle/input', '**', pattern), recursive=True))

hits = find('train_cadences.npy')
assert hits, 'attach the competition data (train_cadences.npy not found under /kaggle/input)'
DATA = os.path.dirname(hits[0])
WORK = '/kaggle/working'
train_meta = pd.read_csv(os.path.join(DATA, 'train_meta.csv'))
test_meta = pd.read_csv(os.path.join(DATA, 'test_meta.csv'))
y = train_meta.label.map(C2I).values.astype(np.int64)
N, NT = len(y), len(test_meta)
SRC_PRIOR = np.bincount(y, minlength=K) / N
log(f'data {DATA} | train {N} | test {NT}')

## 1. Load both teams' out-of-fold and test probabilities
Our models: Model A (canvas ResNet) and Model C (LightGBM), with **PKS-simulated** OOF rows (the proxy that matched the leaderboard twice).
Piyush's models: every `oofpks_<tag>.npy` that has a matching `test_<tag>.npy`.
Rows are matched by position and checked against `train_meta` / `test_meta`.

In [ ]:
npz_paths = find('oof_and_test_probs.npz')
assert npz_paths, 'attach the dataset that contains oof_and_test_probs.npz (the v2 output)'
Z = np.load(npz_paths[0], allow_pickle=True)
ours_cls = [str(c) for c in Z['classes']]
PERM = [ours_cls.index(c) for c in CLASSES]

def oc(a):
    """our arrays -> CLASSES column order"""
    return np.asarray(a, dtype=np.float32)[:, PERM]

assert (Z['test_ids'].astype(str) == test_meta.id.values.astype(str)).all(), 'test order mismatch'
assert (np.array(ours_cls)[Z['y']] == train_meta.label.values).all(), 'train order mismatch'
a_test_keys = sorted(k for k in Z.files if k.startswith('A_test_'))
if CONFIG['OUR_A_TEST'] == 'ens' and a_test_keys:
    a_test = np.mean([oc(Z[k]) for k in a_test_keys], 0)
else:
    a_test = oc(Z['A_test_cv5'] if 'A_test_cv5' in Z.files else Z['A_test'])
MODELS = {}       # name -> dict(oof, test, kind, owner)
MODELS['ours_A'] = dict(oof=oc(Z['A_pks']), test=a_test, kind='cnn', owner='ours')
MODELS['ours_C'] = dict(oof=oc(Z['C_pks']), test=oc(Z['C_test']), kind='gbdt', owner='ours')
if 'B_pks' in Z.files:
    MODELS['ours_B'] = dict(oof=oc(Z['B_pks']), test=oc(Z['B_test']), kind='cnn', owner='ours')
OUR_FEAT_NAMES = [str(s) for s in Z['feat_names']]
F_PKS, F_TE = Z['F_pks'].astype(np.float32), Z['F_te'].astype(np.float32)
log(f'ours: Model A test = mean of {a_test_keys if CONFIG["OUR_A_TEST"] == "ens" else "CV models"} | features {F_PKS.shape[1]}')

for p in find('oofpks_*.npy'):
    tag = os.path.basename(p)[len('oofpks_'):-len('.npy')]
    if 'partial' in tag or tag in CONFIG['EXCLUDE_TAGS']:
        continue
    kind = 'stack' if tag.startswith('stack') else 'cnn'
    if kind == 'stack' and not CONFIG['USE_PIY_STACK']:
        continue
    tps = find(f'test_{tag}.npy')
    if not tps:
        log(f'skip {tag}: no test_{tag}.npy'); continue
    oof, tst = np.load(p).astype(np.float32), np.load(tps[0]).astype(np.float32)
    if oof.shape != (N, K) or tst.shape != (NT, K):
        log(f'skip {tag}: shapes {oof.shape} {tst.shape}'); continue
    if not (np.isfinite(oof).all() and np.isfinite(tst).all()) or oof.sum(1).min() < 0.5:
        log(f'skip {tag}: non-finite or unfinished rows'); continue
    f = mf1(y, oof.argmax(1))
    if f < CONFIG['MIN_MODEL_F1']:
        log(f'skip {tag}: PKS OOF macro-F1 {f:.4f} < {CONFIG["MIN_MODEL_F1"]}'); continue
    MODELS[f'piy_{tag}'] = dict(oof=oof, test=tst, kind=kind, owner='piyush')
n_piy = sum(m['owner'] == 'piyush' for m in MODELS.values())
if n_piy == 0:
    print('\n' + '!' * 90 + '\nNO PIYUSH PROBABILITIES FOUND. The stack will only use our models.\n'
          'Attach his cache/ folder (oofpks_*.npy + test_*.npy) to get the team gain.\n' + '!' * 90 + '\n')
log(f'models loaded: {list(MODELS)}')

### Piyush's level and track features
Loaded from his cache when present (`lvl_trainpks2` / `trk_trainpks2` are the train rows with his PKS-like noise v2).
If the level files are missing they are recomputed here with his exact code (numpy, about 3 min).
Track features need his `cnn_r2g.py`, so they are skipped if missing (our 180 features already contain track searches).

In [ ]:
from scipy.ndimage import uniform_filter1d

def lvl_feats(X):
    """Piyush level_feats.py: ON-panel brightening relative to the OFF bandpass (scintillation)."""
    out = []
    for j in range(0, len(X), 1000):
        x = X[j:j + 1000].astype(np.float32); x[x == 0] = np.nan; x[..., 64] = np.nan
        off = x[:, [1, 3, 5]].reshape(len(x), 48, 128); bO = np.nanmedian(off, 1)
        bO = np.where(np.isnan(bO), np.nanmedian(bO, -1, keepdims=True), bO)
        s = 1.4826 * np.nanmedian(np.abs(off - np.nanmedian(off, 1, keepdims=True)), (1, 2)) + 1e-3
        pm = np.nanmedian(x, 2)
        pm = np.where(np.isnan(pm), bO[:, None, :], pm)
        ex = (pm - bO[:, None, :]) / s[:, None, None]
        lvl = np.median(ex, 2)
        exs = uniform_filter1d(ex, 16, axis=2)
        mx = exs.max(2); frac = (exs > 1.0).mean(2)
        on, of = [0, 2, 4], [1, 3, 5]
        f = np.c_[lvl, mx, frac, lvl[:, on].mean(1) - lvl[:, of].mean(1), lvl[:, on].min(1) - lvl[:, of].max(1),
                  mx[:, on].mean(1) - mx[:, of].mean(1), mx[:, on].min(1), frac[:, on].mean(1) - frac[:, of].mean(1),
                  (ex[:, on] - ex[:, of]).std(2).mean(1)]
        out.append(f)
    return np.concatenate(out).astype(np.float32)

def pks_noisy(x, rng, sig=0.13):
    """Piyush feats_pks2.py: Gamma k=12 power noise + per-panel gain offsets."""
    m = x > 0
    g = np.maximum(rng.gamma(12., 1 / 12., size=x.shape), 1e-4); x = np.clip(x + 24 * np.log2(g), 1, 255)
    xs = np.where(m, x, np.nan); s = 1.4826 * np.nanmedian(np.abs(xs - np.nanmedian(xs, (1, 2, 3), keepdims=True)), (1, 2, 3))
    off = rng.normal(0, sig, size=(len(x), 6, 1, 1)) * s[:, None, None, None]
    return np.where(m, np.clip(x + off, 1, 255), 0).astype(np.float32)

def load_first(names):
    for n in names:
        r = find(n)
        if r:
            return np.load(r[0]).astype(np.float32), n
    return None, None

PIY_FEATS = {}
if CONFIG['USE_PIY_FEATS']:
    lt, ltn = load_first(['lvl_trainpks2.npy', 'lvl_trainpks.npy'])
    le, _ = load_first(['lvl_test.npy'])
    if lt is None or le is None or len(lt) != N or len(le) != NT:
        log('level features not in cache: computing them (Piyush code)')
        Xtr = np.load(os.path.join(DATA, 'train_cadences.npy'), mmap_mode='r')
        Xte = np.load(os.path.join(DATA, 'test_cadences.npy'), mmap_mode='r')
        rng = np.random.default_rng(12)
        lt = np.concatenate([lvl_feats(pks_noisy(Xtr[j:j + 2000].astype(np.float32), rng)) for j in range(0, N, 2000)])
        le = lvl_feats(Xte); ltn = 'computed'
    PIY_FEATS['lvl'] = (lt, le)
    tt, ttn = load_first(['trk_trainpks2.npy', 'trk_trainpks.npy'])
    tte, _ = load_first(['trk_test.npy'])
    if tt is not None and tte is not None and len(tt) == N and len(tte) == NT:
        PIY_FEATS['trk'] = (tt, tte)
    log(f'Piyush features: level from {ltn} {lt.shape} | track {"from " + ttn if "trk" in PIY_FEATS else "missing (skipped)"}')

## 2. Every model on the PKS-simulated OOF, plus simple team averages
Reminder of calibration: our PKS-sim OOF predicted public 0.9029 (v1) and about 0.91 (v2); Piyush's "PKS-like v2" predicted 0.9003 → 0.9003 and 0.9162 → 0.9158.
Both proxies track the leaderboard to within about ±0.005.

In [ ]:
def logp(p):
    return np.log(np.clip(p, 1e-6, 1.0))

def geo_mean(ps, ws=None):
    ws = np.ones(len(ps)) if ws is None else np.asarray(ws, float)
    s = sum(w * logp(p) for w, p in zip(ws, ps)) / ws.sum()
    e = np.exp(s - s.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

rows = []
for name, m in MODELS.items():
    pc = per_class(y, m['oof'].argmax(1))
    rows.append(dict(model=name, owner=m['owner'], kind=m['kind'], pks_oof=mf1(y, m['oof'].argmax(1)),
                     **{s: round(v, 3) for s, v in zip(SHORT, pc)},
                     test_eti=int((m['test'].argmax(1) == 0).sum()), test_scint=int((m['test'].argmax(1) == 3).sum())))
base_table = pd.DataFrame(rows).sort_values('pks_oof', ascending=False).reset_index(drop=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print(base_table.round(4).to_string())

ours_names = [n for n in MODELS if MODELS[n]['owner'] == 'ours']
piy_names = [n for n in MODELS if MODELS[n]['owner'] == 'piyush']
OURS_OOF = geo_mean([MODELS[n]['oof'] for n in ours_names], [0.85 if n == 'ours_A' else 0.15 for n in ours_names])
OURS_TEST = geo_mean([MODELS[n]['test'] for n in ours_names], [0.85 if n == 'ours_A' else 0.15 for n in ours_names])
SIMPLE = {'ours': (OURS_OOF, OURS_TEST)}
if piy_names:
    best_piy = max(piy_names, key=lambda n: mf1(y, MODELS[n]['oof'].argmax(1)))
    PIY_OOF, PIY_TEST = MODELS[best_piy]['oof'], MODELS[best_piy]['test']
    SIMPLE['piyush_best'] = (PIY_OOF, PIY_TEST)
    for w in (0.3, 0.5, 0.7):
        SIMPLE[f'avg_ours{w:.1f}'] = (geo_mean([OURS_OOF, PIY_OOF], [w, 1 - w]), geo_mean([OURS_TEST, PIY_TEST], [w, 1 - w]))
    log(f'best Piyush model: {best_piy}')
for k, (po, pt) in SIMPLE.items():
    log(f'simple {k:14s} PKS OOF macro-F1 {mf1(y, po.argmax(1)):.4f} | test eti {(pt.argmax(1) == 0).sum()} scint {(pt.argmax(1) == 3).sum()}')
if piy_names:
    agree = (OURS_TEST.argmax(1) == PIY_TEST.argmax(1)).mean()
    log(f'test agreement ours vs Piyush best: {100 * agree:.2f}%')

## 3. The team stacker
Inputs per cadence:
* the log-probabilities of every model (train rows = PKS-simulated OOF; test rows = real test predictions)
* our 180 physics features (train rows computed on PKS-simulated cadences)
* Piyush's level and track features (train rows computed on his PKS-like-v2 cadences)

5-fold LightGBM × 3 seeds with early stopping.
The fold models are averaged for the test predictions, so no refit is needed.

In [ ]:
blocks_tr, blocks_te, col_names = [], [], []
for name, m in MODELS.items():
    blocks_tr.append(logp(m['oof'])); blocks_te.append(logp(m['test']))
    col_names += [f'{name}:{s}' for s in SHORT]
if CONFIG['USE_OUR_FEATS']:
    blocks_tr.append(F_PKS); blocks_te.append(F_TE); col_names += [f'our:{n}' for n in OUR_FEAT_NAMES]
for k, (ftr, fte) in PIY_FEATS.items():
    blocks_tr.append(ftr); blocks_te.append(fte); col_names += [f'piy_{k}:{i}' for i in range(ftr.shape[1])]
X_TR = np.column_stack(blocks_tr).astype(np.float32)
X_TE = np.column_stack(blocks_te).astype(np.float32)
assert X_TR.shape[1] == X_TE.shape[1] == len(col_names)
log(f'stack matrix train {X_TR.shape} test {X_TE.shape}')

PRM = dict(objective='multiclass', num_class=K, learning_rate=CONFIG['LR'], num_leaves=CONFIG['LEAVES'],
           min_data_in_leaf=CONFIG['MIN_LEAF'], feature_fraction=CONFIG['FEAT_FRAC'], bagging_fraction=0.8,
           bagging_freq=1, lambda_l2=1.0, verbose=-1, num_threads=os.cpu_count())

def run_stack(Xtr, Xte, tag, seeds=CONFIG['SEEDS']):
    oof, pte, its = np.zeros((N, K)), np.zeros((NT, K)), []
    gain = np.zeros(Xtr.shape[1])
    for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=7).split(Xtr, y)):
        for s in seeds:
            dtr, dva = lgb.Dataset(Xtr[a], y[a]), lgb.Dataset(Xtr[b], y[b])
            mdl = lgb.train(dict(PRM, seed=s), dtr, CONFIG['MAX_ROUNDS'], valid_sets=[dva],
                            callbacks=[lgb.early_stopping(CONFIG['EARLY_STOP'], verbose=False)])
            oof[b] += mdl.predict(Xtr[b], num_iteration=mdl.best_iteration) / len(seeds)
            pte += mdl.predict(Xte, num_iteration=mdl.best_iteration) / (5 * len(seeds))
            its.append(mdl.best_iteration); gain += mdl.feature_importance('gain')
        log(f'{tag} fold {k}: macro-F1 {mf1(y[b], oof[b].argmax(1)):.4f} (iters {its[-len(seeds):]})')
    log(f'{tag}: OOF macro-F1 {mf1(y, oof.argmax(1)):.4f}')
    return oof, pte, gain

STACK_OOF, STACK_TEST, GAIN = run_stack(X_TR, X_TE, 'stack[all]')
pc = per_class(y, STACK_OOF.argmax(1))
print('per class:', dict(zip(SHORT, pc.round(3))))
imp = pd.Series(GAIN, index=col_names).sort_values(ascending=False)
print('top 25 inputs by gain:\n', (imp / imp.sum()).head(25).round(4).to_string())
grp = imp.groupby([c.split(':')[0] for c in imp.index]).sum()
print('gain share by input group:\n', (grp / grp.sum()).sort_values(ascending=False).round(3).to_string())

cm = confusion_matrix(y, STACK_OOF.argmax(1), labels=list(range(K)))
print('stack OOF confusion (rows = true):\n', pd.DataFrame(cm, index=SHORT, columns=SHORT).to_string())

## 4. Decision layer and submissions
* **EM prior (Saerens 2002).** The test telescope has a different label mix. EM estimates the test prior from the stacker's own test posteriors and reweights them.
* **Per-class logit biases.** Tuned on the stack OOF by coordinate ascent on macro-F1.
  They are kept only if tuning on one cross-fitted half also improves the other half (both directions).
  This guards against fitting noise in rare classes.

Files written (in upload order):
1. `submission.csv`: stack + EM (+ biases if they pass the cross-fit check). **Upload first.**
2. `sub_v3_stack_raw.csv`: stack without EM.
3. `sub_v3_stack_em_nobias.csv`: stack + EM without biases (identical to `submission.csv` when the biases are rejected).
4. `sub_v3_avg.csv`: plain geometric average of ours and Piyush's best (no stacker). A robust fallback.

In [ ]:
def em_prior(p, src_prior, iters=300, tol=1e-7):
    p = np.clip(p, 1e-9, 1); pi = src_prior.copy(); q = p
    for _ in range(iters):
        q = p * (pi / src_prior); q = q / q.sum(1, keepdims=True)
        new = q.mean(0)
        if np.abs(new - pi).max() < tol:
            pi = new; break
        pi = new
    return q, pi

def tune_bias(p, yy, grid=CONFIG['BIAS_GRID'], passes=3):
    lp = logp(p); b = np.zeros(K); best = mf1(yy, lp.argmax(1))
    for _ in range(passes):
        for c in range(K):
            for v in grid:
                bb = b.copy(); bb[c] = v
                f = mf1(yy, (lp + bb).argmax(1))
                if f > best + 1e-6:
                    best, b = f, bb
    return b, best

h = np.random.default_rng(0).random(N) < 0.5
gains = []
for tr_mask in (h, ~h):
    b_half, _ = tune_bias(STACK_OOF[tr_mask], y[tr_mask])
    ev = ~tr_mask
    gains.append(mf1(y[ev], (logp(STACK_OOF[ev]) + b_half).argmax(1)) - mf1(y[ev], STACK_OOF[ev].argmax(1)))
BIAS, f_bias = tune_bias(STACK_OOF, y)
USE_BIAS = min(gains) > 0.0
log(f'bias cross-fit gains {np.round(gains, 4)} -> {"ACCEPT" if USE_BIAS else "reject"} | full-OOF tuned F1 {f_bias:.4f} '
    f'vs raw {mf1(y, STACK_OOF.argmax(1)):.4f} | bias {dict(zip(SHORT, BIAS.round(2)))}')
if not USE_BIAS:
    BIAS = np.zeros(K)

P_RAW = STACK_TEST
P_EM, PI_TEST = em_prior(P_RAW, SRC_PRIOR)
P_FINAL = P_EM
LAB_FINAL = (logp(P_FINAL) + BIAS).argmax(1)

def write_sub(labels, fname):
    sub = pd.DataFrame({'id': test_meta.id.values, 'label': np.array(CLASSES)[labels]})
    assert len(sub) == NT and sub.id.is_unique and sub.label.isin(CLASSES).all()
    sub.to_csv(os.path.join(WORK, fname), index=False)
    return sub

write_sub(LAB_FINAL, 'submission.csv')
write_sub(P_EM.argmax(1), 'sub_v3_stack_em_nobias.csv')
write_sub(P_RAW.argmax(1), 'sub_v3_stack_raw.csv')
if 'avg_ours0.5' in SIMPLE:
    write_sub(SIMPLE['avg_ours0.5'][1].argmax(1), 'sub_v3_avg.csv')
log('submissions written')

dist = pd.DataFrame({'train_frac': SRC_PRIOR, 'EM_test_prior': PI_TEST,
                     'stack_raw': np.bincount(P_RAW.argmax(1), minlength=K) / NT,
                     'submission': np.bincount(LAB_FINAL, minlength=K) / NT,
                     'ours_simple': np.bincount(OURS_TEST.argmax(1), minlength=K) / NT}, index=CLASSES)
if piy_names:
    dist['piyush_best'] = np.bincount(PIY_TEST.argmax(1), minlength=K) / NT
print(dist.round(4).to_string())
diff = lambda a, b: f'{100 * (a != b).mean():.2f}%'
log(f'labels changed vs stack_raw {diff(LAB_FINAL, P_RAW.argmax(1))} | vs ours {diff(LAB_FINAL, OURS_TEST.argmax(1))}'
    + (f' | vs Piyush best {diff(LAB_FINAL, PIY_TEST.argmax(1))}' if piy_names else ''))

## 5. Pseudo-labels for the student run, and saved outputs
A test cadence becomes a pseudo-label only if all of these hold:
* the stack (after EM) is at least 90% confident
* our own blend picks the same class
* Piyush's best model picks the same class

Agreement between two independently built systems gives far cleaner labels than one model's confidence alone.
`pl.npz` uses Piyush's format (`sel` = boolean mask over the 10,000 test rows, `y` = class index in the same order), so his `cnn_r2.py ... PL` reads it directly.

In [ ]:
conf = P_FINAL.max(1)
sel = (conf >= CONFIG['PL_THR']) & (LAB_FINAL == OURS_TEST.argmax(1))
if piy_names:
    sel &= LAB_FINAL == PIY_TEST.argmax(1)
y_pl = LAB_FINAL[sel].astype(np.int64)
np.savez(os.path.join(WORK, 'pl.npz'), sel=sel, y=y_pl)
log(f'pseudo-labels: {sel.sum()} of {NT} | per class {dict(zip(SHORT, np.bincount(y_pl, minlength=K)))}')

np.savez_compressed(os.path.join(WORK, 'v3_stack_probs.npz'), classes=np.array(CLASSES), test_ids=test_meta.id.values.astype(str),
                    stack_oof=STACK_OOF.astype(np.float32), stack_test_raw=P_RAW.astype(np.float32),
                    stack_test_em=P_EM.astype(np.float32), bias=BIAS, em_prior=PI_TEST, models=np.array(list(MODELS)))
summary = dict(models=list(MODELS), stack_oof_macro_f1=mf1(y, STACK_OOF.argmax(1)),
               stack_oof_per_class=dict(zip(CLASSES, per_class(y, STACK_OOF.argmax(1)).round(4).tolist())),
               simple=dict((k, mf1(y, po.argmax(1))) for k, (po, pt) in SIMPLE.items()),
               bias_used=bool(USE_BIAS), bias=dict(zip(CLASSES, BIAS.round(3).tolist())),
               em_prior=dict(zip(CLASSES, PI_TEST.round(4).tolist())), n_pseudo=int(sel.sum()),
               minutes=round((time.time() - T0) / 60, 1))
json.dump(summary, open(os.path.join(WORK, 'v3_summary.json'), 'w'), indent=1)

print('=' * 100)
print('SUMMARY  (decision number = PKS-sim OOF macro-F1; it has matched the public leaderboard within about 0.005)')
print(base_table[['model', 'owner', 'pks_oof']].round(4).to_string(index=False))
for k, (po, pt) in SIMPLE.items():
    print(f'  simple {k:16s} {mf1(y, po.argmax(1)):.4f}')
print(f'  TEAM STACK             {mf1(y, STACK_OOF.argmax(1)):.4f}   <- submission.csv (+ EM{", + biases" if USE_BIAS else ""})')
print('files:', sorted(f for f in os.listdir(WORK) if f.endswith(('.csv', '.npz', '.json'))))
log('done')